# ES 335 ML Project: 01 Dataset Inspection & Data Cleaning Audit

This notebook audits the raw **UCI Online Retail dataset** (`Online Retail.xlsx`), identifies missing values, duplicates, cancellations, non-positive values, and non-product codes, and verifies the 5 sequential cleaning rules.

In [1]:
import os
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.cleaning import audit_raw_data, clean_online_retail_data

# Load raw transactions
raw_path = '../data/raw/Online Retail.xlsx'
print(f'Loading {raw_path}...')
df_raw = pd.read_excel(raw_path)
print(f'Raw dataset shape: {df_raw.shape}')
df_raw.info()

Loading ../data/raw/Online Retail.xlsx...


Raw dataset shape: (541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


## 1. Initial Data Audit (Before Cleaning)

In [2]:
audit_stats = audit_raw_data(df_raw)
total = audit_stats['total_rows']
print(f'Total raw rows: {total:,}')
print(f'Duplicate rows: {audit_stats["duplicates_count"]:,}')
print(f'Cancellations (InvoiceNo starting with C): {audit_stats["cancellations_count"]:,}')
print(f'Non-positive Quantity (<= 0): {audit_stats["non_pos_qty_count"]:,}')
print(f'Non-positive UnitPrice (<= 0): {audit_stats["non_pos_price_count"]:,}')
print(f'Non-product StockCodes: {audit_stats["non_product_codes_count"]:,}')
print('\nMissing values per column:')
for col, cnt in audit_stats['missing_counts'].items():
    print(f'  {col:15s}: {cnt:7,d} missing ({cnt/total:.2%})')

Total raw rows: 541,909
Duplicate rows: 5,268
Cancellations (InvoiceNo starting with C): 9,288
Non-positive Quantity (<= 0): 10,624
Non-positive UnitPrice (<= 0): 2,517
Non-product StockCodes: 2,950

Missing values per column:
  InvoiceNo      :       0 missing (0.00%)
  StockCode      :       0 missing (0.00%)
  Description    :   1,454 missing (0.27%)
  Quantity       :       0 missing (0.00%)
  InvoiceDate    :       0 missing (0.00%)
  UnitPrice      :       0 missing (0.00%)
  CustomerID     : 135,080 missing (24.93%)
  Country        :       0 missing (0.00%)


## 2. Apply Sequential Cleaning Rules & Verify Removal Counts

In [3]:
clean_df, removal_logs = clean_online_retail_data(df_raw, verbose=True)
print(f'\nFinal Cleaned Rows: {len(clean_df):,} ({len(clean_df)/total:.2%} retained)')
removal_logs


2. APPLYING CLEANING RULES (SEQUENTIAL REMOVAL COUNTS)


• Rule 1 (Remove Duplicates):
    Removed :   5,268 rows
    Remaining: 536,641 rows

• Rule 2 (Remove Cancellations starting with 'C'):
    Removed :   9,251 rows
    Remaining: 527,390 rows

• Rule 3 (Remove Non-positive Quantity <= 0):
    Removed :   1,336 rows
    Remaining: 526,054 rows

• Rule 4 (Remove Non-positive UnitPrice <= 0):
    Removed :   1,176 rows
    Remaining: 524,878 rows



• Rule 5 (Remove Non-product StockCodes):
    Removed :   2,341 rows
    Remaining: 522,537 rows


3. CLEANING SUMMARY
Initial raw rows  : 541,909
Total rows removed:  19,372 (3.57%)
Final clean rows  : 522,537 (96.43%)
Unique products   :   3,907
Date range        : 2010-12-01 08:26:00 to 2011-12-09 12:50:00

• Post-cleaning null counts:
    - InvoiceNo      :       0
    - StockCode      :       0
    - Description    :       0
    - Quantity       :       0
    - InvoiceDate    :       0
    - UnitPrice      :       0
    - CustomerID     : 131,387
    - Country        :       0
    - Week_Start     :       0

Final Cleaned Rows: 522,537 (96.43% retained)


,Rule,Rows_Removed,Rows_Remaining,Pct_Dataset_Removed
0,Rule 1: Remove Exact Duplicates,5268,536641,0.972119
1,Rule 2: Remove Cancellations (InvoiceNo starti...,9251,527390,1.707113
2,Rule 3: Remove Non-positive Quantity (Quantity...,1336,526054,0.246536
3,Rule 4: Remove Non-positive UnitPrice (UnitPri...,1176,524878,0.217011
4,"Rule 5: Remove Non-product StockCodes (POST, D...",2341,522537,0.431991


### Key Takeaways from Data Inspection
- Initial raw records: **541,909**
- Exactly 5,268 duplicate rows removed
- Exactly 9,251 cancellations ('C') removed
- Exactly 1,336 non-positive quantity rows removed
- Exactly 1,176 non-positive unit price rows removed
- Exactly 2,341 non-product service/fee codes removed ('POST', 'M', 'D', 'CRUK', etc.)
- Final cleaned dataset: **522,537 rows** across **3,907 unique products**.